In [1]:
# -----------------------------
# 1. Setup
# -----------------------------
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr
import statsmodels.api as sm

# Example tickers + sector info
tickers = ["AAPL", "MSFT", "AMZN", "GOOG", "META", "JPM", "GS", "XOM", "CVX", "TSLA"]
sector_map = {}
for t in tickers:
    info = yf.Ticker(t).info
    sector_map[t] = info.get("sector", "Unknown")
sector_df = pd.Series(sector_map, name="Sector")

# Download data
data = yf.download(tickers, start="2020-01-01", end="2023-01-01")["Close"]


/var/folders/h2/r7qn2m9n1zb6y_0q191gdqth0000gn/T/ipykernel_9355/918082388.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start="2020-01-01", end="2023-01-01")["Close"]
[*********************100%***********************]  10 of 10 completed


In [2]:
# -----------------------------
# 2. Create factor (20-day momentum)
# -----------------------------
returns = data.pct_change()
factor = (data / data.shift(20) - 1).dropna(how='all')   # 20-day momentum factor
fwd_returns = returns.shift(-1)       # forward return for IC

In [3]:
# -----------------------------
# 3. Compute daily IC
# -----------------------------
ICs = []
for t in factor.index:
    f = factor.loc[t].dropna()        # factor values on date t
    r = fwd_returns.loc[t].reindex(f.index).dropna() # forward returns on date t
    if len(f) > 2 and len(r) == len(f):
        ICs.append(spearmanr(f, r).correlation)    # IC is correlation of factor and forward return
ICs = pd.Series(ICs, index=factor.index[:len(ICs)])

mean_IC = ICs.mean()
IR = mean_IC / ICs.std()

print("Raw Factor Results")
print(f"Mean IC: {mean_IC:.4f}, IR: {IR:.4f}")


Raw Factor Results
Mean IC: 0.0237, IR: 0.0527


In [8]:
row - row.mean()

array([[ 0.09359067,  0.49241665, -1.20600583,  0.43887563, -0.04356658,
        -0.68339022, -0.25336796,  0.57652358,  1.86028484, -1.27536077]])

In [9]:
# -----------------------------
# 3. PCA on factor loadings (systematic risk)
# -----------------------------
# Assume `factor` is a DataFrame: index=dates, columns=stocks
factor_std = (factor - factor.mean()) / factor.std()
factor_neutralized_sys = pd.DataFrame(index=factor.index, columns=factor.columns)

window = 20  # rolling window length in days

for date in factor_std.index[window-1:]:
    # take last `window` days of factor values
    window_data = factor_std.loc[:date].tail(window)  # shape: (window, n_stocks)
    
    if window_data.isna().any().any():
        factor_neutralized_sys.loc[date] = np.nan
        continue

    # compute covariance across stocks over the window
    cov_matrix = np.cov(window_data.T)  # shape: (n_stocks, n_stocks)

    # PCA on covariance
    pca = PCA(n_components=1)  # remove only first PC
    pca.fit(cov_matrix)
    pc1 = pca.components_.T  # shape: (n_stocks, 1)

    # project out first PC from today's factor
    row = factor_std.loc[date].values.reshape(1, -1)
    row_centered = row - row.mean()
    projection = (row_centered @ pc1) @ pc1.T
    factor_neutralized_sys.loc[date] = (row_centered - projection).flatten()

In [10]:
factor_neutralized_sys

Ticker,AAPL,AMZN,CVX,GOOG,GS,JPM,META,MSFT,TSLA,XOM
Date,,,,,,,,,,
2020-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-02-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-02-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2022-12-23,-0.670163,-0.115742,0.278323,-0.397478,-0.47892,0.253356,1.255983,0.043535,-0.891872,0.217911
2022-12-27,-0.583501,-0.43113,0.566567,-0.474249,-0.460243,0.416485,1.369532,0.225018,-1.282089,0.518133
2022-12-28,-0.579352,-0.322733,0.406242,-0.459206,-0.421617,0.394516,1.320142,0.269359,-1.077575,0.412543


In [ ]:
# -----------------------------
# 4. Neutralization (industry + size)
# -----------------------------
mktcap = data * 1e6  # fake market cap proxy (price * 1m shares)

neutralized = pd.DataFrame(index=factor.index, columns=factor.columns)

for t in factor.index:
    f = factor.loc[t].dropna()
    if len(f) < 3: 
        continue
    
    # Features: industry dummies + log(MarketCap)
    sec = sector_df.reindex(f.index)
    dummies = pd.get_dummies(sec)
    size = np.log(mktcap.loc[t, f.index])
    X = pd.concat([dummies, 
                size.rename('log_size'), 
                (size**2).rename('log_size2')], axis=1).fillna(0)
    
    model = LinearRegression().fit(X, f)
    fitted = model.predict(X)
    neutralized.loc[t, f.index] = f - fitted


In [ ]:
# -----------------------------
# 5. Recompute IC after neutralization
# -----------------------------
ICs_neut = []
for t in neutralized.index:
    f = neutralized.loc[t].dropna()
    r = fwd_returns.loc[t].reindex(f.index).dropna()
    if len(f) > 2 and len(r) == len(f):
        ICs_neut.append(spearmanr(f, r).correlation)
ICs_neut = pd.Series(ICs_neut, index=neutralized.index[:len(ICs_neut)])

mean_IC_neut = ICs_neut.mean()
IR_neut = mean_IC_neut / ICs_neut.std()

print("\nNeutralized Factor Results")
print(f"Mean IC: {mean_IC_neut:.4f}, IR: {IR_neut:.4f}")


Neutralized Factor Results
Mean IC: 0.0070, IR: 0.0246
